In [15]:
EXTRACT_USER_INFO_PROMPT = """

You are a highly accurate resume information extraction system.

Your ONLY task is to extract the primary person's identity and contact
information from the provided resume content.

The input may contain ONE resume or MULTIPLE resume versions belonging
to the same person. All resume content is provided as ONE string.

You MUST follow the extraction and validation rules below.

============================================================
PRIMARY PERSON
============================================================

Extract information ONLY for the primary person whose resume is being
provided.

Do NOT extract information belonging to:

- Recruiters
- Hiring managers
- Interviewers
- References
- Professors
- Managers
- Colleagues
- Clients
- Companies
- Organizations
- Other people mentioned inside the resume

If multiple resume versions clearly belong to the same person, combine
their information.

If information belongs to different people, DO NOT merge their
information.

============================================================
VERY IMPORTANT: DO NOT GUESS
============================================================

NEVER invent, infer, guess, or fabricate a value.

Every returned value must be supported by the resume content.

If a field cannot be identified with reasonable confidence,
return an empty string.

NEVER move a value from one field to another just because another
field is empty.

For example:

If an email address is found but a portfolio URL is not found:

CORRECT:
portfolio_url = ""
email = "person@example.com"

INCORRECT:
portfolio_url = "person@example.com"

============================================================
FULL NAME
============================================================

Extract the person's full name.

The strongest evidence for the full name is:

1. Name at the beginning/header of the resume.
2. Name in the contact section.
3. Name explicitly associated with the resume.

Example:

# Anirban Das

=> full_name = "Anirban Das"

Do NOT use:

- Admission numbers
- Employee IDs
- Student IDs
- Usernames
- GitHub usernames
- LinkedIn usernames
- Names of companies
- Names of professors
- Names of recruiters
- Names mentioned in projects

============================================================
PHONE
============================================================

Extract ALL phone numbers that belong to the primary person.

IMPORTANT:

The output MUST be a STRING.

If multiple phone numbers exist, separate them using comma + space.

Example:

"+91 9876543210, +91 9123456789"

DO NOT return an array.

Only extract values that are clearly phone numbers.

Valid phone numbers may contain:

- Country code
- Spaces
- Hyphens
- Parentheses
- Digits

Examples:

+91 9876543210
+91-9876543210
9876543210
+1 415 555 1234

Do NOT extract:

- Admission numbers
- Student IDs
- Employee IDs
- Roll numbers
- Application IDs
- GitHub usernames
- LeetCode usernames
- Ratings
- Random numeric values
- Dates
- CGPA
- Years
- Company numbers
- Recruiter phone numbers

IMPORTANT:

A value such as:

23JE0104

is NOT a phone number.

A value such as:

6290375587

may be a phone number if it appears in the person's contact
information.

============================================================
EMAIL
============================================================

Extract ALL email addresses belonging to the primary person.

The output MUST be a STRING.

If multiple emails exist, separate them using comma + space.

Example:

"john@example.com, john.doe@gmail.com"

DO NOT return an array.

An email MUST contain a valid email-like structure:

local-part@domain

For example:

dasanirban268@gmail.com

is an email.

IMPORTANT:

An email address MUST NEVER be returned as:

- linkedin_url
- github_url
- portfolio_url

If a value contains "@", it is an email candidate, NOT a URL.

Do NOT extract:

- Recruiter emails
- Company/general emails
- Reference emails
- Emails belonging to other people

============================================================
LINKEDIN URL
============================================================

Extract the PRIMARY PERSON'S LinkedIn profile URL.

A valid LinkedIn value must identify LinkedIn.

Examples:

https://www.linkedin.com/in/anirbandas

https://linkedin.com/in/anirbandas

linkedin.com/in/anirbandas

All of the above are valid LinkedIn URLs.

If the protocol is missing, you MAY preserve the URL as it appears
in the resume.

For example:

linkedin.com/in/anirbandas

is acceptable.

IMPORTANT VALIDATION:

The value MUST contain:

linkedin.com

and preferably:

linkedin.com/in/

Do NOT put these into linkedin_url:

- Email addresses
- GitHub URLs
- Portfolio URLs
- Company URLs
- Plain usernames unless clearly identified as LinkedIn

If no LinkedIn URL is found:

linkedin_url = ""

============================================================
GITHUB URL
============================================================

Extract the PRIMARY PERSON'S GitHub profile URL.

Examples:

https://github.com/anirban2005143a

https://github.com/anirban2005143a/

github.com/anirban2005143a

All are valid GitHub URLs.

IMPORTANT VALIDATION:

The value MUST contain:

github.com

A GitHub URL should normally look like:

github.com/<username>

If the resume contains:

github.com/anirban2005143a

return:

github.com/anirban2005143a

If the resume contains:

https://github.com/anirban2005143a/project

and it is clearly the person's repository, you MAY extract the
profile URL:

https://github.com/anirban2005143a

IMPORTANT:

Do NOT put these into github_url:

- Email addresses
- LinkedIn URLs
- Portfolio URLs
- GitHub repository URLs belonging to another person
- Random GitHub links mentioned in job descriptions

If no GitHub URL is found:

github_url = ""

============================================================
PORTFOLIO URL
============================================================

Extract the person's PERSONAL PORTFOLIO WEBSITE URL.

This field is ONLY for a personal website/portfolio.

Valid examples:

https://anirban-das-portfolio.vercel.app/

https://anirbandas.dev

https://anirbandas.com

https://anirban.github.io

IMPORTANT VALIDATION:

A portfolio_url MUST be a URL.

It MUST NOT be:

- An email address
- A phone number
- A LinkedIn URL
- A GitHub URL
- A company website
- A university website
- A job board
- A social media profile

VERY IMPORTANT:

If the value contains "@":

IT IS NOT A PORTFOLIO URL.

For example:

dasanirban268@gmail.com

MUST NEVER be returned as portfolio_url.

If no personal portfolio website is present:

portfolio_url = ""

============================================================
URL CLASSIFICATION
============================================================

When extracting URLs, classify them using these rules:

IF value contains:
"linkedin.com"

=> linkedin_url

IF value contains:
"github.com"

=> github_url

IF value is a valid HTTP/HTTPS/web URL and is clearly the person's
personal website/portfolio:

=> portfolio_url

IF value contains "@":

=> email

NEVER assign the same value to multiple fields.

For example:

https://linkedin.com/in/anirbandas

MUST NOT appear in:

github_url
portfolio_url
email

Similarly:

dasanirban268@gmail.com

MUST NOT appear in:

linkedin_url
github_url
portfolio_url

============================================================
CONTACT HEADER EXTRACTION
============================================================

Resume contact information is often written in a compact format.

For example:

Name | phone | email | LinkedIn | GitHub | Portfolio

or:

Name
phone
email
LinkedIn
GitHub
portfolio website

Do NOT rely only on the order of these fields.

Instead, classify each candidate using its actual format.

Example:

Anirban Das | 6290375587 | dasanirban268@gmail.com |
linkedin.com/in/anirbandas |
github.com/anirban2005143a |
https://anirban-das-portfolio.vercel.app/

Correct extraction:

full_name:
"Anirban Das"

phone:
"6290375587"

email:
"dasanirban268@gmail.com"

linkedin_url:
"linkedin.com/in/anirbandas"

github_url:
"github.com/anirban2005143a"

portfolio_url:
"https://anirban-das-portfolio.vercel.app/"

============================================================
MULTIPLE RESUME VERSIONS
============================================================

The input may contain:

RESUME 1
RESUME 2
RESUME 3
...

These may be different versions of the same person's resume.

If they clearly belong to the same person:

1. Combine their contact information.
2. Remove duplicate phone numbers.
3. Remove duplicate email addresses.
4. Select the best LinkedIn URL.
5. Select the best GitHub URL.
6. Select the best personal portfolio URL.

For phone numbers:

If Resume 1 contains:

6290355877

and Resume 2 contains:

6290375587

return:

"6290355877, 6290375587"

For emails:

If Resume 1 contains:

person@gmail.com

and Resume 2 contains:

person@outlook.com

return:

"person@gmail.com, person@outlook.com"

DO NOT merge information belonging to different people.

============================================================
DEDUPLICATION
============================================================

Remove exact duplicates.

Example:

Resume 1:
person@gmail.com

Resume 2:
person@gmail.com

Output:

"person@gmail.com"

NOT:

"person@gmail.com, person@gmail.com"

Same rule applies to phone numbers.

============================================================
MISSING VALUES
============================================================

If information is not present, return:

""

Do NOT use:

null
None
N/A
Not found
Unknown
Not available

============================================================
FIELD TYPE REQUIREMENTS
============================================================

The output MUST contain exactly these fields:

full_name
phone
linkedin_url
github_url
portfolio_url
email

All six fields MUST be strings.

phone:
comma-separated string if multiple numbers exist.

email:
comma-separated string if multiple emails exist.

All URL fields:
single string.

============================================================
FINAL VALIDATION BEFORE OUTPUT
============================================================

Before returning the answer, internally validate every field.

CHECK 1:
Does full_name look like a person's name?

CHECK 2:
Does phone contain only phone-number candidates?

CHECK 3:
Does linkedin_url contain "linkedin.com"?

CHECK 4:
Does github_url contain "github.com"?

CHECK 5:
Is portfolio_url actually a web URL?

CHECK 6:
Does portfolio_url NOT contain "@"

CHECK 7:
Does email contain valid email candidates?

CHECK 8:
Is the same value incorrectly assigned to multiple fields?

CHECK 9:
Are duplicate phones removed?

CHECK 10:
Are duplicate emails removed?

If any field fails validation, correct it before returning the output.

============================================================
ADDITIONAL USER INSTRUCTION
============================================================

{user_instruction}

============================================================
RESUME CONTENT
============================================================

{resume_content}

============================================================
OUTPUT FORMAT
============================================================

{format_instructions}

============================================================
FINAL OUTPUT RULE
============================================================

Return ONLY the structured output.

Do not provide explanations.

Do not provide reasoning.

Do not provide Markdown.

Do not provide comments.

Do not provide additional fields.

Do not write anything before or after the structured output.

"""

In [16]:
from pydantic import BaseModel, Field


class UserInformation(BaseModel):
    full_name: str = Field(
        default="",
        description="The full name of the person."
    )

    phone: str = Field(
        default="",
        description=(
            "All phone numbers belonging to the person, "
            "as a comma-separated string. Empty string if not found."
        )
    )

    linkedin_url: str = Field(
        default="",
        description="The person's LinkedIn profile URL. Empty string if not found."
    )

    github_url: str = Field(
        default="",
        description="The person's GitHub profile URL. Empty string if not found."
    )

    portfolio_url: str = Field(
        default="",
        description="The person's personal portfolio URL. Empty string if not found."
    )

    email: str = Field(
        default="",
        description=(
            "All email addresses belonging to the person, "
            "as a comma-separated string. Empty string if not found."
        )
    )

In [17]:
import os

from dotenv import load_dotenv

from langchain_huggingface import (
    HuggingFaceEndpoint,
    ChatHuggingFace,
)

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_classic.output_parsers import OutputFixingParser

# from .extract_user_info_prompt import EXTRACT_USER_INFO_PROMPT
# from .extract_user_info_schema import UserInformation


load_dotenv()


llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token=os.environ["HUGGINGFACEHUB_API_TOKEN1"],
    max_new_tokens=1024,
    temperature=0.1,
)

model = ChatHuggingFace(llm=llm)


parser = PydanticOutputParser(
    pydantic_object=UserInformation
)


fixing_parser = OutputFixingParser.from_llm(
    parser=parser,
    llm=model,
    max_retries=2,
)


prompt = ChatPromptTemplate.from_template(
    EXTRACT_USER_INFO_PROMPT
)


extract_user_info_chain = prompt | model | fixing_parser


def extract_user_information(
    resume_content: str,
    user_instruction: str = "",
) -> UserInformation:

    if not resume_content or not resume_content.strip():
        raise ValueError("resume_content must not be empty")

    result = extract_user_info_chain.invoke(
        {
            "resume_content": resume_content.strip(),
            "user_instruction": user_instruction.strip(),
            "format_instructions": parser.get_format_instructions(),
        }
    )

    return result

In [18]:
resume = """
====================
RESUME 1
====================
# **Anirban Das** 

(+91) 6290375587 _|_ dasaniran268@gmail.com _|_ LinkedIn _|_ GitHub 

## **Education** 

### **Indian Institute of Technology (ISM), Dhanbad, India** 

Bachelor of Technology (CGPA: 8.29/10) 2023 – **Relevant Coursework:** Data Structures & Algorithms, Object Oriented Programming (C++), Operating Systems, Database Management Systems (DBMS), Compiler Design 

2023 – 2027 

## **Experience** 

### **Google** _|_ **Software Engineer Intern** 

   - May 2026 – Jul 2026 

- Engineered a new linting rule and automated batch validation pipeline in Java and TypeScript to detect structural errors across cloud contract templates. 

- Enhanced core template management interfaces by developing a responsive document comparison dialog box, a read/write mode selector for gDoc, and an embedded AI assistant chat widget. 

- Contributed 7,350+ lines of production code across 22 peer-reviewed changelists and created analytical dashboards for contract monitoring and template health tracking. 

## **Projects** 

**Code Fusion** _|_ **React.js, MongoDB, Yjs, Socket.IO, ExpressJS** 

Deployed _|_ GitHub 

- Built a CRDT-based real-time collaborative code editor using Yjs and Monaco Editor, enabling conflict-free multi-user editing through delta-based synchronization and awareness protocol for live cursor tracking across sessions. 

- Developed a real-time collaboration system supporting live cursor tracking and integrated in-app chat, allowing seamless communication between distributed developers during shared coding sessions. 

- Implemented multi-language support, auto-completion, real-time syntax error detection, persistent code storage, customizable themes, and a collapsible file explorer for efficient workflow management. 

### **JobPilot** _|_ **FastAPI, Next.js, Socket.IO, LangChain, Tailwind CSS** 

   - GitHub 

- Developed an automation system using FastAPI, Next.js, and HuggingFace LLMs to automate resume extraction and job matching, reducing manual application time by 80%. 

- Architected a scalable asynchronous worker system with WebSockets/Socket.IO updates and FileLock synchronization to manage concurrent multi-user job lifecycles. 

- Built an LLM-powered job ranking and categorization pipeline using prompt engineering, along with a manual review workflow and local validation setup. 

## **Competitive Programming** 

**LeetCode:** aswU2SZvDg _|_ Solved 240+ problems focusing on data structures and algorithms **CodeForces:** anirban2005 _|_ Max Rating: 1216 (Pupil) _|_ Solved 390+ problems 

## **Technical Skills** 

- **Languages & Databases:** C++, Python, JavaScript, TypeScript, MongoDB, PostgreSQL, Vector Databases 

- **Core Frameworks:** React.js, Next.js, Node.js, Express.js, FastAPI, Tailwind CSS, Three.js 

- **AI/ML & Tools:** LangChain, LangGraph, TensorFlow, Keras, Git, Docker, Postman 

## **Achievements** 

- Secured **4th** rank at **HaXplore** | **CodeFest’25** , organized by **IIT BHU** . 

- **Winner** of Winter of Code 6.0 (Web Development Division), a one-month long hackathon conducted by **CyberLabs** , IIT (ISM) Dhanbad. | **Deployed Project** 


====================
RESUME 2
====================
# **Anirban Das** 

**Adm. No.** 23JE0104 � 6290375587 � My <u>portfolio website-https://anirban-das-portfolio.vercel.app/</u>
� dasanirban268@gmail.com � <u>linkedin.com/in/anirbandas</u> � <u>github.com/anirban2005143a</u> 



## Education 

### **Indian Institute of Technology (Indian School of Mines), Dhanbad** 

_Bachelor of Technology in Computer Science and Engineering (GPA: 8.29 / 10.00)_ 

Expected May 2027 _Dhanbad, Jharkhand_ 

- **Relevant Coursework:** Data Structures and Algorithms (C++), Database Management System, Compiler Design, Computer Organization , Computer Architecture, Operating Systems. 

## Experience 

### **Google** _|_ **_Software Engineer Intern_** 

May 2026 – July 2026 

- Engineered a new linting rule and automated batch validation pipeline in Java and TypeScript to detect structural errors across cloud contract templates. 

- Enhanced core template management interfaces by developing a responsive document comparison dialog box, a read/write mode selector for gDoc, and an embedded AI assistant chat widget. 

- Contributed 7,350+ lines of production code across 22 peer-reviewed changelists and created analytical dashboards for contract monitoring and template health tracking. 

## Projects 

**<u>Code Fusion</u>** _| React.js, Flask, Express.js, MongoDB, Tailwind CSS |_ _<u>GitHub</u> | Deployed Project_ 

- An online code editor supporting real-time collaboration, multiple languages, and customizable themes. 

- Enables multiple developers to collaborate in real-time with live cursor tracking and integrated in-app chat for seamless communication. 

- Integrates auto-completion, real-time syntax error detection, and persistent code-saving functionality, while offering various themes, multi-language support, and a collapsible sidebar for efficient file management. 

**<u>JobPilot</u>** _| Python (FastAPI), TypeScript (Next.js), Langchain, WebSockets |_ _<u>GitHub</u> |_ _<u>Video</u>_ 

- Developed a full-stack automation system using **FastAPI** , **Next.js** , and **HuggingFace LLMs** to automate resume extraction and job matching, reducing manual application time by **80%** . 

- Architected a scalable background worker system with **asynchronous processing** , **WebSockets** for real-time updates, and **FileLock** synchronization to manage concurrent multi-user job lifecycles. 

- Built an **LLM-powered** pipeline using **prompt engineering** for job ranking and categorization, featuring a manual review workflow and a local mock portal for safe system validation. 

**NoteBridge** _| React.js, Express.js, MongoDB, Bootstrap |_ _<u>GitHub</u> | Deployed Project_ 

- A **feature-rich note-taking and sharing platform** that enables **structured organization** through folders and facilitates **controlled file sharing** . 

- Enables **interactive engagement** through features like **likes** , **comments** , and **shares** . 

- Provides a **comprehensive profile page** displaying total posts, followers, following, and an **organized archive of past posts** for easy access and engagement. 

## Technical Skills 

**AI/ML & Agents** : LangChain, LangGraph, TensorFlow, Keras, Deep Learning, ANN, CNN, LSTM. **Technologies** : Node.js, FastAPI, Express.js, Docker, Next.js, React.js, Tailwind CSS, Three.js, GSAP. **Database & Cloud** : MongoDB, PostgreSQL, Vector Databases. 

## Achievements 

- Secured **4th** rank at **HaXplore** _|_ **CodeFest’25** , organized by **IIT BHU!** 

- **Winner** - of Winter Of Code 6.O (in Web Development Division) a one-month long hackathon conducted by **CyberLabs** , IIT(ISM) Dhanbad. _| Deployed Project_ 

## Social Engagements 

- Member of CyberLabs -Tech society of IIT ISM Dhanbad 

- Member of Aquatics Team - Swimming Team of IIT ISM Dhanbad. 

- Represented IIT Dhanbad at the 37th INTER IIT AQUATICS MEET 2023 held at IIT Gandhinagar and secured **4th place** in 200m Individual Medley . 


"""

In [19]:
result = extract_user_information(
    resume_content=resume,
)

print(result.model_dump())

{'full_name': 'Anirban Das', 'phone': '6290375587', 'linkedin_url': 'linkedin.com/in/anirbandas', 'github_url': 'github.com/anirban2005143a', 'portfolio_url': 'https://anirban-das-portfolio.vercel.app/', 'email': 'dasanirban268@gmail.com'}
